# Healthcare fairness audit — replication on a synthetic fixture

Runs `fairscope`'s `HealthcareFairnessAudit` on the **synthetic** fixture committed in the repo (`tests/fixtures/healthcare_subsample.csv`). The fixture is *not* real BRFSS data; it is seed-generated to reproduce the **direction and approximate magnitude** of a published age-disparity finding (elderly < young AUC; diabetes study, IEEE CIPHER 2026). It demonstrates the audit pipeline, not an empirical result.

In [ ]:
import matplotlib

matplotlib.use('Agg')
from pathlib import Path

import pandas as pd

from fairscope.healthcare import HealthcareFairnessAudit

In [ ]:
fixture = Path('..') / 'tests' / 'fixtures' / 'healthcare_subsample.csv'
if not fixture.exists():
    fixture = Path('tests') / 'fixtures' / 'healthcare_subsample.csv'
df = pd.read_csv(fixture, comment='#')
df.head()

In [ ]:
report = HealthcareFairnessAudit.from_scores(
    df['y_true'].to_numpy(),
    df['y_score'].to_numpy(),
    {'age_group': df['age_group'].to_numpy()},
).run()
report.to_dataframe()

Per-subgroup AUC with DeLong confidence intervals, ECE, Brier, and F1. The summary flags the largest gap and any Bonferroni-significant difference.

In [ ]:
print(report.summary())

In [ ]:
report.plot_auc_forest()

In [ ]:
report.plot_calibration()

The audit recovers the published finding's **direction** (elderly AUC < young) on the synthetic fixture. The published p < 0.001 was obtained on the full ~1.28M-row cohort; see `tests/fixtures/README.md`.